# PKG customer coverage: RM, banker, TMO and segment per power id

Output `bdahd01p_dlcdi1_cdi_tm.pkg_cust_coverage`: one row per power id, **current state** (rebuilt whole on each run). For filtering apps (deposit attrition) by officer and segment; not a model input. Joins to every PKG table on `customer_pwr_id`. `pkg_cust_dim` is not touched (metrics and attrition caches fingerprint it).

Sources (CDM, `clhhd01p_clh`)
- `cust_pwr_id_curnt_dim_keys`: current RM, lead RM, business banking banker and lead, domestic TMO keys; `cust_dim_key` + `max_time_key` point to the customer's latest customer fact row.
- `cdm_customer_fact_v` at that row: `sales_gl_cc_hrchy_key` (the customer's sales cost center, as in the VECTOR notebook), `tmo_intl_empl_dim_key`, and `party_dim_key` → `cdm_party_dim_v.party_id` = **mdm_id** (as in VECTOR's MDM query).
- `cdm_employee_dim_v`, `cdm_gl_cc_prim_org_hierarchy_dim_v`, `cdm_customer_dim_v`: versioned dimensions; `active_flg = Y` marks the current version of a row (checked 2026-11-10), not an active person or customer.

Rules
- A key from a fact or key table points to one version. Decode: key → that version (any version, **no active filter**) → natural id (`empl_id`, `cntr_nbr`) → its current version (active Y, newest `eff_dt`). Names, titles and hierarchy are today's; an active-only join loses every key whose version has since been replaced (counted in cell 4).
- Officer status: `assigned`; `left` = the employee's current version has `purge_flg = Y` (left the bank, `_left_dt` = `purge_dt`); `placeholder` = key below -1 (PNC Admin, Migration Manager, MDM Admin …, name kept as the label); `none` = key -1 or missing; `unknown key` = positive key not in the employee dimension.
- Segment columns from the current version of the cost center: lob = roll-up level 2, segment = level 3, sub_segment = level 4, team = level 5, market, legacy_bank (acquisition node).
- Power ids listed twice in the key table: newest `max_time_key`, then a real RM key, then the highest `cust_dim_key`.
- `cust_purged_flg` / `_dt` from the customer dimension's current version (customers who left stay in the table).

In [ ]:
# ============================================================
# 1. Setup and config
# ============================================================
import os, re, html, warnings, datetime as _dt
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore")

# --- NumPy alias shim (old pyspark code paths use removed aliases) ---
import numpy as np
for _a, _t in (("object0", object), ("bool8", np.bool_), ("int0", np.intp), ("uint0", np.uintp),
               ("bool", bool), ("object", object)):
    if not hasattr(np, _a):
        setattr(np, _a, _t)

# --- pandas shim: PySpark 3.3 createDataFrame calls iteritems (removed in pandas 2.0) ---
import pandas as pd
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items

from IPython.display import display, HTML
from pyspark.sql import SparkSession, Window, functions as F, types as T
from pyspark import StorageLevel

spark = (SparkSession.builder
         .appName("pkg_cust_coverage")
         .config("spark.ui.showConsoleProgress", "false")   # no stage progress bars in cell outputs
         .enableHiveSupport()
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
try:  # silence Atlas lineage hook + harmless executor-release noise (log4j2 on Spark 3.3)
    _L = spark._jvm.org.apache.logging.log4j
    for n in ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage",
              "org.apache.kafka", "org.apache.spark.sql.catalyst.util.package",
              "org.apache.spark.network", "io.netty"):
        _L.core.config.Configurator.setLevel(n, _L.Level.OFF)
except Exception:
    pass
# Unparseable date strings -> NULL instead of SparkUpgradeException (we probe many formats)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

# ---------------- config ----------------
DB          = "bdahd01p_dlcdi1_cdi_tm"
T_COV       = "pkg_cust_coverage"            # one row per power id, current state
KEYS        = "clhhd01p_clh.cust_pwr_id_curnt_dim_keys"   # current officer keys + pointer to the latest fact row
CUST_FACT   = "clhhd01p_clh.cdm_customer_fact_v"          # sales cost center + international TMO on that row
CUST_DIM    = "clhhd01p_clh.cdm_customer_dim_v"           # customer purge flag (current version)
EMPL_DIM    = "clhhd01p_clh.cdm_employee_dim_v"
GLCC_DIM    = "clhhd01p_clh.cdm_gl_cc_prim_org_hierarchy_dim_v"
PARTY_DIM   = "clhhd01p_clh.cdm_party_dim_v"                # party_dim_key on the fact row -> party_id = mdm_id
PKG_DIM     = f"{DB}.pkg_cust_dim"           # only for the in_pkg flag (or the scope when SCOPE = "pkg")
SCOPE       = "all"     # "all" = every power id in KEYS (in_pkg flags PKG customers); "pkg" = power ids in pkg_cust_dim only
SEG_FROM_TIME_KEY = 20230101   # segment / intl TMO read only from fact rows with time_key >= this (older last rows: no segment)
HDFS_REPLICATION = 2
MIN_FREE_TB = 0.2
VERSION     = "cust_coverage_v1"
OUT_DIR     = "../metrics/cust_coverage"
os.makedirs(OUT_DIR, exist_ok=True)
spark.sparkContext._jsc.hadoopConfiguration().set("dfs.replication", str(HDFS_REPLICATION))
spark.conf.set("dfs.replication", str(HDFS_REPLICATION))

# ---------------- helpers ----------------
CSS = """<style>
.cx {font-family: Arial, sans-serif; font-size: 12px;}
.cx h3 {margin: 14px 0 4px 0; font-size: 14px;}
.cx h4 {margin: 10px 0 3px 0; font-size: 13px; color: #1f3b63;}
.cx table {border-collapse: collapse; margin: 2px 0 8px 0;}
.cx th {background: #e8eef6; border: 1px solid #c8d2e0; padding: 2px 6px; text-align: left; white-space: nowrap;}
.cx td {border: 1px solid #dde3ec; padding: 2px 6px; vertical-align: top; max-width: 420px; overflow-wrap: anywhere;}
.cx .note {color: #555; margin: 2px 0;}
.cx .warn {color: #a33; font-weight: bold;}
.cx pre {background: #f6f8fa; border: 1px solid #dde3ec; padding: 6px; white-space: pre-wrap; font-size: 11px; max-width: 1400px;}
</style>"""

def esc(x):
    return html.escape("" if x is None else str(x))

def tbl(pdf, max_rows=None):
    """pandas DataFrame -> compact HTML table (values escaped)."""
    if pdf is None or len(pdf) == 0:
        return "<p class='note'>(no rows)</p>"
    if max_rows is not None:
        pdf = pdf.head(max_rows)
    head = "".join(f"<th>{esc(c)}</th>" for c in pdf.columns)
    body = "".join("<tr>" + "".join(f"<td>{esc(v)}</td>" for v in row) + "</tr>"
                   for row in pdf.astype(object).where(pdf.notna(), "").itertuples(index=False))
    return f"<table><tr>{head}</tr>{body}</table>"

def show(parts):
    """One consolidated HTML output per cell; tex2jax_ignore keeps any dollar sign from turning into math."""
    display(HTML(CSS + "<div class='cx tex2jax_ignore mathjax_ignore'>" + "".join(parts) + "</div>"))

def fnum(x):
    if x is None:
        return ""
    try:
        x = float(x)
    except Exception:
        return str(x)
    if np.isnan(x):
        return ""
    a = abs(x)
    for d, s in ((1e12, "T"), (1e9, "B"), (1e6, "M"), (1e3, "K")):
        if a >= d:
            return f"{x / d:.2f}{s}"
    if x == int(x):
        return str(int(x))
    return f"{x:.4g}"

def fpct(x, digits=1):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    return f"{100 * float(x):.{digits}f}%"

def qcol(c):
    return F.col(f"`{c}`")

NUMERIC_T = (T.ByteType, T.ShortType, T.IntegerType, T.LongType, T.FloatType, T.DoubleType, T.DecimalType)
COMPLEX_T = (T.ArrayType, T.MapType, T.StructType, T.BinaryType)
DATE_NAME = re.compile(r"(_dt$|_dte$|_date$|^date_|_date_|_ts$|_tmstmp|timestamp|^dt_|_dt_|asof|as_of|snap|_month$|_mo$|_yrmo|yyyymm|period)")

def kind_of(dtype):
    if isinstance(dtype, (T.DateType, T.TimestampType)): return "date"
    if isinstance(dtype, T.BooleanType): return "bool"
    if isinstance(dtype, NUMERIC_T): return "num"
    if isinstance(dtype, COMPLEX_T): return "complex"
    return "str"

def is_date_like(name, dtype):
    k = kind_of(dtype)
    if k == "date":
        return True
    return k in ("str", "num") and bool(DATE_NAME.search(name.lower())) \
        and not isinstance(dtype, (T.FloatType, T.DoubleType))

def date_expr(c, dtype):
    """Parse a date-like column of any common storage format to DATE (NULL if unparseable)."""
    col = qcol(c)
    if isinstance(dtype, T.DateType):
        return col
    if isinstance(dtype, T.TimestampType):
        return F.to_date(col)
    s = F.trim(col.cast("string"))
    return (F.when(s.rlike(r"^\d{4}-\d{2}-\d{2}"), F.to_date(F.substring(s, 1, 10), "yyyy-MM-dd"))
             .when(s.rlike(r"^\d{8}$"), F.to_date(s, "yyyyMMdd"))
             .when(s.rlike(r"^\d{1,2}/\d{1,2}/\d{4}"),
                   F.to_date(F.regexp_extract(s, r"^(\d{1,2}/\d{1,2}/\d{4})", 1), "M/d/yyyy"))
             .when(s.rlike(r"^\d{6}$"), F.to_date(F.concat(s, F.lit("01")), "yyyyMMdd"))
             .when(s.rlike(r"^\d{4}-\d{2}$"), F.to_date(F.concat(s, F.lit("-01")), "yyyy-MM-dd")))

def real_date(p):
    """Drop placeholder dates (year <= 1900 or >= 2100, e.g. 0001-01-01 / 9999-12-31) for min/max."""
    return F.when(F.year(p).between(1901, 2099), p)

show([f"<h3>Setup</h3><p>Spark {esc(spark.version)} · pandas {esc(pd.__version__)} · numpy {esc(np.__version__)}</p>",
      tbl(pd.DataFrame([("Output", f"{DB}.{T_COV}"), ("Scope", SCOPE), ("Officer keys", KEYS), ("Fact", CUST_FACT),
                        ("Employee dim", EMPL_DIM), ("Cost center dim", GLCC_DIM), ("Customer dim", CUST_DIM), ("Party dim (mdm_id)", PARTY_DIM),
                        ("SEG_FROM_TIME_KEY", SEG_FROM_TIME_KEY), ("HDFS_REPLICATION", HDFS_REPLICATION),
                        ("VERSION", VERSION), ("OUT_DIR", os.path.abspath(OUT_DIR))], columns=["setting", "value"]))])


In [ ]:
# ============================================================
# 2. Inputs: columns, key uniqueness, scope, quota
# ============================================================
import json, uuid, time

def clean_pwr(c):
    """Power id as 10-digit text; '-1', blanks, zeros and non-digits -> NULL."""
    s = F.trim(F.col(c).cast("string"))
    s = F.when(s.rlike(r"^[0-9]{1,10}$"), F.lpad(s, 10, "0"))
    return F.when(s != F.lit("0000000000"), s)

_jvm, _hconf = spark._jvm, spark._jsc.hadoopConfiguration()
def _fp(p):
    P = _jvm.org.apache.hadoop.fs.Path(p)
    return P.getFileSystem(_hconf), P
def fs_write(p, text):
    fs, P = _fp(p)
    out = fs.create(P, True)
    out.write(bytearray(text.encode("utf-8")))
    out.close()
def fs_delete(p):
    fs, P = _fp(p)
    if fs.exists(P):
        fs.delete(P, True)
def fs_rename(a, b):
    fs, A = _fp(a)
    _, B = _fp(b)
    if not fs.rename(A, B):
        raise RuntimeError(f"rename failed: {a} -> {b}")
def quota(path):
    """(folder holding the space quota, quota bytes, consumed bytes), counted with copies."""
    fs, P = _fp(path)
    q = P
    while q is not None:
        try:
            cs = fs.getContentSummary(q)
            if cs.getSpaceQuota() >= 0:
                return str(q), cs.getSpaceQuota(), cs.getSpaceConsumed()
        except Exception:
            pass
        q = q.getParent()
    return None, None, fs.getContentSummary(P).getSpaceConsumed()
def space_used(path):
    fs, P = _fp(path)
    return fs.getContentSummary(P).getSpaceConsumed() if fs.exists(P) else 0
def set_replication(path, n):
    fs, P = _fp(path)
    if not fs.exists(P):
        return 0
    it, k = fs.listFiles(P, True), 0
    while it.hasNext():
        st = it.next()
        if st.getReplication() > n and fs.setReplication(st.getPath(), n):
            k += 1
    return k

# ---- columns (names from the 2026-11-10 profiles and the VECTOR notebook) ----
ROLE_KEYS = {"rm": "curnt_rm_empl_dim_key", "rm_lead": "curnt_rm_lead_empl_dim_key",
             "bb_banker": "curnt_bb_bk_empl_dim_key", "bb_banker_lead": "curnt_bb_bk_lead_empl_dim_key",
             "tmo_dom": "curnt_tmo_dom_empl_dim_key"}
NEED = {KEYS: ["cust_pwr_id", "cust_dim_key", "max_time_key"] + list(ROLE_KEYS.values()),
        CUST_FACT: ["cust_dim_key", "time_key", "sales_gl_cc_hrchy_key", "tmo_intl_empl_dim_key", "party_dim_key"],
        PARTY_DIM: ["party_dim_key", "party_id"],
        CUST_DIM: ["cust_pwr_id", "purge_flg", "purge_dt", "eff_dt", "active_flg"],
        EMPL_DIM: ["empl_dim_key", "empl_id", "empl_user_id", "empl_full_name", "empl_job_title", "manager_desc",
                   "empl_work_state", "purge_flg", "purge_dt", "eff_dt", "active_flg"],
        GLCC_DIM: ["gl_cc_hrchy_key", "cntr_nbr", "cntr_desc", "gl_cc_roll_up_level_2_desc", "gl_cc_roll_up_level_3_desc",
                   "gl_cc_roll_up_level_4_desc", "gl_cc_roll_up_level_5_desc", "market_desc",
                   "acquisition_hrchy_node_dsc", "eff_dt", "active_flg"],
        PKG_DIM: ["customer_pwr_id", "mdm_id"]}
SRC = {t: spark.table(t) for t in NEED}
miss = {t: [c for c in cols if c not in SRC[t].columns] for t, cols in NEED.items()}
miss = {t: m for t, m in miss.items() if m}
if miss:
    raise RuntimeError(f"Missing columns: {miss}")
if SCOPE not in ("all", "pkg"):
    raise RuntimeError("SCOPE must be 'all' or 'pkg'")

# ---- dimension keys must be unique (one version per key) ----
for t, k in ((EMPL_DIM, "empl_dim_key"), (GLCC_DIM, "gl_cc_hrchy_key"), (PARTY_DIM, "party_dim_key")):
    r = SRC[t].agg(F.count(F.lit(1)).alias("n"), F.countDistinct(k).alias("d")).collect()[0]
    if r["n"] != r["d"]:
        raise RuntimeError(f"{t}.{k} is not unique: {r['n']:,} rows, {r['d']:,} keys")

# ---- scope ----
PKG_IDS = (SRC[PKG_DIM].select(clean_pwr("customer_pwr_id").alias("customer_pwr_id"))
                       .filter("customer_pwr_id is not null").distinct().withColumn("in_pkg", F.lit(1)))
k0 = SRC[KEYS].withColumn("customer_pwr_id", clean_pwr("cust_pwr_id")).filter("customer_pwr_id is not null")
kc = k0.agg(F.count(F.lit(1)).alias("rows"), F.countDistinct("customer_pwr_id").alias("ids")).collect()[0]
n_pkg = PKG_IDS.count()

# ---- location and quota ----
_d = spark.sql(f"DESCRIBE DATABASE EXTENDED {DB}").toPandas()
LOC = str(_d.loc[_d.iloc[:, 0].astype(str).str.strip().str.lower() == "location"].iloc[0, 1]).rstrip("/")
PATH = f"{LOC}/{T_COV}"
QF, QB, QU = quota(LOC)
TB = 1e12
QTXT = (f"{QU / TB:.2f} of {QB / TB:.2f} TB used, {(QB - QU) / TB:.2f} TB free (quota on {QF})" if QB is not None
        else f"no space quota found up the tree; folder uses {QU / TB:.2f} TB")
if QB is not None and QB - QU < MIN_FREE_TB * TB:
    raise RuntimeError(f"cdi_tm quota nearly full: {QTXT}")

show(["<h3>Inputs</h3>", tbl(pd.DataFrame([
    ("Key table rows / power ids", f"{kc['rows']:,} / {kc['ids']:,} ({kc['rows'] - kc['ids']:,} extra rows: newest kept)"),
    ("PKG power ids (pkg_cust_dim)", f"{n_pkg:,}"),
    ("Scope", "every power id in the key table" if SCOPE == "all" else "PKG power ids only"),
    ("Employee / cost center / party keys unique", "yes"),
    ("Output folder", PATH),
    ("Quota", QTXT),
    ("Rough need", "under 1 GB with copies (one row per power id)"),
], columns=["item", "value"]))])


In [ ]:
# ============================================================
# 3. Build: officer keys, fact pointer, decode, write
# ============================================================
t0 = time.time()
U = lambda c: F.upper(F.trim(F.col(c).cast("string")))

# ---- 1. one key row per power id ----
w = Window.partitionBy("customer_pwr_id").orderBy(F.col("max_time_key").desc(),
                                                 (F.col("curnt_rm_empl_dim_key") > 0).desc(), F.col("cust_dim_key").desc())
K = (k0.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1")
       .select("customer_pwr_id", F.col("cust_dim_key").cast("long").alias("cust_dim_key"),
               F.col("max_time_key").cast("long").alias("max_time_key"),
               *[F.col(c).cast("long").alias(f"k_{r}") for r, c in ROLE_KEYS.items()])
       .join(PKG_IDS, "customer_pwr_id", "left").withColumn("in_pkg", F.coalesce("in_pkg", F.lit(0))))
if SCOPE == "pkg":
    K = K.filter("in_pkg = 1")
K = K.persist(StorageLevel.MEMORY_AND_DISK)
n_k = K.count()

# ---- 2. the latest fact row: sales cost center + international TMO ----
tks = sorted({int(r[0]) for r in K.filter(F.col("max_time_key") >= SEG_FROM_TIME_KEY).select("max_time_key").distinct().collect()})
if not tks:
    raise RuntimeError(f"No power id has max_time_key >= {SEG_FROM_TIME_KEY}")
FX = (SRC[CUST_FACT].filter(F.col("time_key").isin(tks))
        .select(F.col("cust_dim_key").cast("long").alias("cust_dim_key"), F.col("time_key").cast("long").alias("max_time_key"),
                F.col("sales_gl_cc_hrchy_key").cast("long").alias("seg_key"),
                F.col("tmo_intl_empl_dim_key").cast("long").alias("k_tmo_intl"),
                F.col("party_dim_key").cast("long").alias("party_key"))
        .join(K.select("customer_pwr_id", "cust_dim_key", "max_time_key"), ["cust_dim_key", "max_time_key"])
        # a load may repeat a customer: keep a real cost center first, deterministic
        .groupBy("customer_pwr_id")
        .agg(F.max(F.struct((F.col("seg_key") > 0).cast("int").alias("a"), F.col("seg_key").alias("s"),
                            F.col("k_tmo_intl").alias("t"), F.col("party_key").alias("p"))).alias("x"))
        .select("customer_pwr_id", F.lit(1).alias("fact_found"), F.col("x.s").alias("seg_key"), F.col("x.t").alias("k_tmo_intl"),
                F.col("x.p").alias("party_key")))
PTY = SRC[PARTY_DIM].select(F.col("party_dim_key").cast("long").alias("party_key"),
                             F.trim(F.col("party_id").cast("string")).alias("mdm_id"))
K2 = (K.join(FX, "customer_pwr_id", "left").join(PTY, "party_key", "left")
       .withColumn("mdm_id", F.when(F.col("party_key") > 0, F.col("mdm_id")))
       .persist(StorageLevel.MEMORY_AND_DISK))
ROLES = list(ROLE_KEYS) + ["tmo_intl"]

# ---- 3. employee keys: key -> its version -> empl_id -> current version ----
E = SRC[EMPL_DIM]
we = Window.partitionBy("empl_id").orderBy((U("active_flg") == "Y").desc(), F.col("eff_dt").desc(), F.col("empl_dim_key").desc())
E_CUR = (E.filter(F.col("empl_dim_key") > 0)
          .withColumn("_rn", F.row_number().over(we)).filter("_rn = 1")
          .select("empl_id", F.trim("empl_user_id").alias("user_id"), F.trim("empl_full_name").alias("name"),
                  F.trim("empl_job_title").alias("title"), F.trim("manager_desc").alias("manager_name"),
                  F.trim("empl_work_state").alias("work_state"), (U("purge_flg") == "Y").alias("left"),
                  F.to_date("purge_dt").alias("left_dt")))
used = None
for r in ROLES:
    s = K2.select(F.col(f"k_{r}").alias("k")).filter("k is not null")
    used = s if used is None else used.unionByName(s)
used = used.distinct()
KD = (used.join(E.select(F.col("empl_dim_key").cast("long").alias("k"), "empl_id",
                         (U("active_flg") == "Y").alias("key_active"), F.trim("empl_full_name").alias("key_name")), "k", "left")
          .join(E_CUR, "empl_id", "left")
          .select("k",
                  F.when(F.col("k") == -1, "none").when(F.col("k") <= 0, "placeholder")
                   .when(F.col("key_name").isNull() & F.col("empl_id").isNull(), "unknown key")
                   .when(F.col("left"), "left").otherwise("assigned").alias("status"),
                  F.when(F.col("k") > 0, F.col("user_id")).alias("user_id"),
                  F.when(F.col("k") > 0, F.coalesce("name", "key_name")).when(F.col("k") < -1, F.col("key_name")).alias("name"),
                  "title", "manager_name", "work_state",
                  F.when(F.col("left"), F.col("left_dt")).alias("left_dt"),
                  F.when(F.col("k") > 0, ~F.coalesce("key_active", F.lit(True))).alias("old_version")))
KD = F.broadcast(KD.persist(StorageLevel.MEMORY_AND_DISK))

# ---- 4. segment: key -> its version -> cntr_nbr -> current version ----
G = SRC[GLCC_DIM]
wg = Window.partitionBy("cntr_nbr").orderBy((U("active_flg") == "Y").desc(), F.col("eff_dt").desc(), F.col("gl_cc_hrchy_key").desc())
G_CUR = (G.withColumn("_rn", F.row_number().over(wg)).filter("_rn = 1")
          .select("cntr_nbr", F.trim("gl_cc_roll_up_level_2_desc").alias("lob"), F.trim("gl_cc_roll_up_level_3_desc").alias("segment"),
                  F.trim("gl_cc_roll_up_level_4_desc").alias("sub_segment"), F.trim("gl_cc_roll_up_level_5_desc").alias("team"),
                  F.trim("market_desc").alias("market"), F.trim("acquisition_hrchy_node_dsc").alias("legacy_bank"),
                  F.trim("cntr_nbr").alias("cost_center_nbr"), F.trim("cntr_desc").alias("cost_center_name")))
SD = (K2.select("seg_key").filter("seg_key > 0").distinct()
        .join(G.select(F.col("gl_cc_hrchy_key").cast("long").alias("seg_key"), "cntr_nbr",
                       (U("active_flg") == "Y").alias("seg_key_active")), "seg_key", "left")
        .join(G_CUR, "cntr_nbr", "left").drop("cntr_nbr")
        .withColumn("seg_old_version", ~F.coalesce("seg_key_active", F.lit(True))).drop("seg_key_active"))

# ---- 5. customer purge (current version; active_flg is a partition, so only current rows are read) ----
wc = Window.partitionBy("customer_pwr_id").orderBy(F.col("eff_dt").desc())
CP = (SRC[CUST_DIM].filter(U("active_flg") == "Y")
        .select(clean_pwr("cust_pwr_id").alias("customer_pwr_id"), "eff_dt",
                (U("purge_flg") == "Y").cast("int").alias("cust_purged_flg"), F.to_date("purge_dt").alias("pdt"))
        .filter("customer_pwr_id is not null")
        .withColumn("_rn", F.row_number().over(wc)).filter("_rn = 1")
        .select("customer_pwr_id", "cust_purged_flg",
                F.when(F.col("cust_purged_flg") == 1, F.col("pdt")).alias("cust_purged_dt")))

# ---- 6. assemble: one row per power id ----
OUT = K2
ROLE_COLS = []
for r in ROLES:
    keep = ["user_id", "name", "status", "left_dt", "old_version"] + (["title", "manager_name", "work_state"] if r == "rm" else [])
    sub = KD.select(F.col("k").alias(f"k_{r}"), *[F.col(c).alias(f"{r}_{c}") for c in keep])
    OUT = OUT.join(sub, f"k_{r}", "left").withColumn(f"{r}_status", F.coalesce(F.col(f"{r}_status"), F.lit("none")))
    ROLE_COLS += [f"{r}_{c}" for c in keep if c != "old_version"]
OUT = (OUT.join(F.broadcast(SD), "seg_key", "left").join(CP, "customer_pwr_id", "left")
          .withColumn("fact_last_dt", F.to_date(F.col("max_time_key").cast("string"), "yyyyMMdd"))
          .withColumn("seg_status", F.when(F.col("max_time_key") < SEG_FROM_TIME_KEY, "last fact row too old")
                                     .when(F.col("fact_found").isNull(), "fact row not found")
                                     .when(F.coalesce("seg_key", F.lit(-1)) <= 0, "no cost center")
                                     .when(F.col("cost_center_nbr").isNull(), "unknown key").otherwise("found")))
SEG_COLS = ["lob", "segment", "sub_segment", "team", "market", "legacy_bank", "cost_center_nbr", "cost_center_name"]
FINAL = ["customer_pwr_id", "mdm_id", "in_pkg", "fact_last_dt"] + ROLE_COLS + SEG_COLS + ["seg_status", "cust_purged_flg", "cust_purged_dt"]
QA_COLS = [f"{r}_old_version" for r in ROLES] + ["seg_old_version"]

# ---- 7. write to staging, check, replace ----
STAGE = f"{LOC}/_pkg_cov_staging/run={uuid.uuid4().hex[:8]}"
(OUT.select(*FINAL, *QA_COLS).repartition(16).write.mode("overwrite").parquet(STAGE + "/qa"))
QA = spark.read.parquet(STAGE + "/qa")
chk = QA.agg(F.count(F.lit(1)).alias("n"), F.countDistinct("customer_pwr_id").alias("d")).collect()[0]
if chk["n"] != n_k or chk["d"] != n_k:
    raise RuntimeError(f"Expected one row per power id ({n_k:,}); got {chk['n']:,} rows, {chk['d']:,} ids. Staging kept at {STAGE}")
QA.select(*FINAL).repartition(8).write.mode("overwrite").parquet(STAGE + "/table")
fs_write(STAGE + "/table/_PKG_MARKER.json", json.dumps({"version": VERSION, "scope": SCOPE, "rows": n_k,
                                                        "built": time.strftime("%Y-%m-%d %H:%M:%S")}))
fs_delete(PATH)
fs_rename(STAGE + "/table", PATH)
n_rep = set_replication(PATH, HDFS_REPLICATION) + set_replication(STAGE, HDFS_REPLICATION)

# ---- 8. register (external, not partitioned) ----
FULL = f"{DB}.{T_COV}"
sch = spark.read.parquet(PATH).schema
if spark.sql(f"SHOW TABLES IN {DB} LIKE '{T_COV}'").count() > 0:
    d = spark.sql(f"DESCRIBE TABLE EXTENDED {FULL}").toPandas()
    loc = d.loc[d["col_name"].astype(str).str.strip() == "Location", "data_type"].astype(str).tolist()
    if not loc or loc[0].rstrip("/") != PATH.rstrip("/"):
        raise RuntimeError(f"{FULL} already exists at {loc}; not replacing it")
    spark.sql(f"DROP TABLE {FULL}")
cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in sch.fields)
spark.sql(f"CREATE TABLE {FULL} ({cols}) USING parquet LOCATION '{PATH}'")
K.unpersist(); K2.unpersist()
show([f"<h3>Built {esc(FULL)}</h3>", tbl(pd.DataFrame([
    ("Rows (one per power id)", f"{n_k:,}"),
    ("Fact load dates read", f"{len(tks):,} (time_key >= {SEG_FROM_TIME_KEY})"),
    ("Size with copies", f"{space_used(PATH) / 1e9:.2f} GB"),
    ("Files set to 2 copies", f"{n_rep:,}"),
    ("QA copy (cell 4; delete afterwards)", STAGE + "/qa"),
    ("Minutes", f"{(time.time() - t0) / 60:.1f}"),
], columns=["item", "value"])), f"<p class='note'>In Impala: INVALIDATE METADATA {esc(FULL)};</p>"])


In [ ]:
# ============================================================
# 4. QA: coverage by role, what an active-only join loses, segment mix
# ============================================================
Q = spark.read.parquet(STAGE + "/qa").persist(StorageLevel.MEMORY_AND_DISK)
groups = [("all power ids", F.lit(True)), ("PKG customers", F.col("in_pkg") == 1),
          ("PKG customers, not purged", (F.col("in_pkg") == 1) & (F.coalesce("cust_purged_flg", F.lit(0)) == 0))]

# officer status by role
rows = []
for gname, cond in groups:
    sub = Q.filter(cond)
    n = sub.count()
    for r in ROLES:
        st = {x[0]: x[1] for x in sub.groupBy(f"{r}_status").count().collect()}
        ov = sub.filter(F.col(f"{r}_old_version")).count()
        rows.append({"customers": gname, "role": r, "n": f"{n:,}",
                     **{s: fpct(st.get(s, 0) / n) if n else "" for s in ("assigned", "left", "placeholder", "none", "unknown key")},
                     "key only in an old version": fpct(ov / n) if n else ""})
ROLE_T = pd.DataFrame(rows)

# segment status
srows = []
for gname, cond in groups:
    sub = Q.filter(cond)
    n = sub.count()
    st = {x[0]: x[1] for x in sub.groupBy("seg_status").count().collect()}
    ov = sub.filter(F.col("seg_old_version") == True).count()
    srows.append({"customers": gname, **{s: fpct(st.get(s, 0) / n) if n else "" for s in
                  ("found", "no cost center", "unknown key", "fact row not found", "last fact row too old")},
                  "key only in an old version": fpct(ov / n) if n else ""})
SEG_T = pd.DataFrame(srows)

# mdm_id: filled, and agreement with pkg_cust_dim (same power id -> same mdm_id?)
pkm = (SRC[PKG_DIM].select(clean_pwr("customer_pwr_id").alias("customer_pwr_id"), F.trim(F.col("mdm_id").cast("string")).alias("pkg_mdm"))
                   .filter("customer_pwr_id is not null"))
mm = (Q.select("customer_pwr_id", "mdm_id", "in_pkg").join(pkm, "customer_pwr_id", "left")
        .groupBy("customer_pwr_id", "mdm_id", "in_pkg")
        .agg(F.max((F.col("pkg_mdm") == F.col("mdm_id")).cast("int")).alias("same"), F.count("pkg_mdm").alias("n_pkg_mdm")))
mr = mm.agg(F.count(F.lit(1)).alias("n"), F.sum(F.col("mdm_id").isNotNull().cast("int")).alias("filled"),
            F.sum("in_pkg").alias("n_pkg"),
            F.sum(F.when((F.col("in_pkg") == 1) & (F.col("same") == 1), 1).otherwise(0)).alias("same"),
            F.sum(F.when((F.col("in_pkg") == 1) & F.col("mdm_id").isNotNull() & (F.coalesce("same", F.lit(0)) == 0), 1).otherwise(0)).alias("differ"),
            F.sum(F.when(F.col("n_pkg_mdm") > 1, 1).otherwise(0)).alias("multi")).collect()[0]
MDM_T = pd.DataFrame([
    ("Power ids with an mdm_id", f"{mr['filled']:,} of {mr['n']:,} ({fpct(mr['filled'] / mr['n']) if mr['n'] else ''})"),
    ("PKG power ids: mdm_id matches one in pkg_cust_dim", f"{mr['same']:,} of {mr['n_pkg']:,}"),
    ("PKG power ids: mdm_id filled but not among pkg_cust_dim's", f"{mr['differ']:,}"),
    ("Power ids with 2+ mdm_ids in pkg_cust_dim", f"{mr['multi']:,}"),
    ("Power ids not in pkg_cust_dim (no PKG payments)", f"{mr['n'] - mr['n_pkg']:,}"),
], columns=["check", "value"])

pk = Q.filter("in_pkg = 1")
n_pk = pk.count()
def top(c, k=15):
    t = pk.groupBy(F.coalesce(F.col(c), F.lit("(none)")).alias(c)).count().orderBy(F.desc("count")).limit(k).toPandas()
    t["share"] = (t["count"] / max(n_pk, 1)).map(fpct)
    t["count"] = t["count"].map(lambda v: f"{v:,}")
    return t
left_rm = (pk.filter("rm_status = 'left'").groupBy(F.year("rm_left_dt").alias("RM left in")).count()
             .orderBy(F.desc("RM left in")).limit(10).toPandas())
ph = (Q.filter("rm_status = 'placeholder'").groupBy("rm_name").count().orderBy(F.desc("count")).toPandas())
SAMPLE = (pk.filter("rm_status = 'assigned' and seg_status = 'found'").select(*FINAL).limit(5).toPandas()
            .astype(str).T.reset_index().rename(columns={"index": "column"}))

DICT = pd.DataFrame([
    ("customer_pwr_id", "Power id, 10 digits (one row each)"),
    ("mdm_id", "party_id of the party on the latest fact row (cdm_party_dim_v), as in VECTOR's MDM query"),
    ("in_pkg", "1 = power id in pkg_cust_dim"),
    ("fact_last_dt", "Date of the customer's latest customer fact row (max_time_key); older = customer dropped out of the fact"),
    ("<role>_user_id, _name", "Officer login and name, from the employee's current version. Roles: rm, rm_lead, bb_banker, bb_banker_lead, tmo_dom (current keys table), tmo_intl (latest fact row)"),
    ("<role>_status", "assigned / left (employee purged: left the bank) / placeholder (PNC Admin, Migration Manager …; name = label) / none (no officer) / unknown key"),
    ("<role>_left_dt", "Date the officer left the bank (purge_dt), when status = left"),
    ("rm_title, rm_manager_name, rm_work_state", "RM's job title, manager and work state (current version)"),
    ("lob, segment, sub_segment, team", "Sales cost center roll-up levels 2, 3, 4, 5 (current version of the cost center)"),
    ("market, legacy_bank", "Market and acquisition node (Legacy PNC, BBVA …) of the sales cost center"),
    ("cost_center_nbr, cost_center_name", "Sales cost center (sales_gl_cc_hrchy_key on the latest fact row)"),
    ("seg_status", "found / no cost center / unknown key / fact row not found / last fact row too old (before SEG_FROM_TIME_KEY)"),
    ("cust_purged_flg, cust_purged_dt", "Customer purged in the customer dimension (current version): no longer a customer"),
], columns=["column", "definition"])
DICT.to_csv(os.path.join(OUT_DIR, "cust_coverage_dictionary.csv"), index=False)
Q.unpersist()
show(["<h3>mdm_id</h3>", tbl(MDM_T), "<h3>Officer coverage by role (share of customers)</h3>",
      "<p class='note'>Last column: the key points to a replaced version, so a join to active_flg = Y rows only (as in VECTOR) leaves it blank.</p>",
      tbl(ROLE_T), "<h3>Segment</h3>", tbl(SEG_T),
      f"<h3>PKG customers ({n_pk:,})</h3>", "<h4>lob</h4>", tbl(top("lob")), "<h4>segment</h4>", tbl(top("segment")),
      "<h4>market</h4>", tbl(top("market")), "<h4>RM status</h4>", tbl(top("rm_status")),
      "<h4>RM left the bank, by year</h4>", tbl(left_rm), "<h4>RM placeholders (all power ids)</h4>", tbl(ph),
      "<h4>Five PKG customers with an RM and a segment</h4>", tbl(SAMPLE),
      "<h4>Columns</h4>", tbl(DICT),
      f"<p class='note'>Dictionary saved: {esc(os.path.abspath(os.path.join(OUT_DIR, 'cust_coverage_dictionary.csv')))}. "
      f"When done, delete the QA copy: fs_delete('{esc(STAGE)}')</p>"])
